# Parte Específica: valores "Unknown" no BankChurners

No BankChurners a ausência não aparece como `NaN`: ela foi registrada como o texto `"Unknown"`
em três variáveis categóricas (`Education_Level`, `Marital_Status`, `Income_Category`).

Este notebook investiga se a ausência depende de outras variáveis (para apoiar a discussão do
mecanismo de ausência) e aplica o tratamento escolhido pelo grupo.

In [1]:
# [IA] Claude Code – importacao das bibliotecas e leitura do dataset
from pathlib import Path

import pandas as pd

CAMINHO_DADOS = Path("../../data/parte_especifica/BankChurners.csv")
df = pd.read_csv(CAMINHO_DADOS)

COLUNAS_COM_UNKNOWN = ["Education_Level", "Marital_Status", "Income_Category"]

## 1. Quantidade de Unknown

In [2]:
# [IA] Claude Code – quantidade e porcentagem de Unknown em cada coluna
resumo_unknown = pd.DataFrame({
    "qtd_unknown": [(df[c] == "Unknown").sum() for c in COLUNAS_COM_UNKNOWN],
}, index=COLUNAS_COM_UNKNOWN)
resumo_unknown["pct_unknown"] = (resumo_unknown["qtd_unknown"] / len(df) * 100).round(2)
resumo_unknown

,qtd_unknown,pct_unknown
Education_Level,1519,15.00
Marital_Status,749,7.40
Income_Category,1112,10.98


In [3]:
# [IA] Claude Code – colunas indicadoras: 1 se o valor e Unknown, 0 se e conhecido
indicadores = pd.DataFrame({
    f"unknown_{c}": (df[c] == "Unknown").astype(int) for c in COLUNAS_COM_UNKNOWN
})
indicadores.sum()

unknown_Education_Level    1519
unknown_Marital_Status      749
unknown_Income_Category    1112
dtype: int64

## 2. Os Unknown aparecem juntos?

In [4]:
# [IA] Claude Code – combinacoes de Unknown nas tres colunas (1 = Unknown)
indicadores.value_counts().rename("qtd_linhas").reset_index()

,unknown_Education_Level,unknown_Marital_Status,unknown_Income_Category,qtd_linhas
0,0,0,0,7081
1,1,0,0,1267
2,0,0,1,892
3,0,1,0,560
4,1,0,1,138
5,1,1,0,107
6,0,1,1,75
7,1,1,1,7


In [5]:
# [IA] Claude Code – taxa de Unknown em cada coluna, separando quem tem ou nao Unknown nas outras
linhas = []
for alvo in COLUNAS_COM_UNKNOWN:
    for outra in COLUNAS_COM_UNKNOWN:
        if outra == alvo:
            continue
        grupo = indicadores[f"unknown_{outra}"]
        taxa = indicadores[f"unknown_{alvo}"].groupby(grupo).mean() * 100
        linhas.append({
            "coluna": alvo,
            "condicao": f"{outra} e Unknown?",
            "pct_unknown_quando_nao": round(taxa.get(0, float("nan")), 2),
            "pct_unknown_quando_sim": round(taxa.get(1, float("nan")), 2),
        })
pd.DataFrame(linhas)

,coluna,condicao,pct_unknown_quando_nao,pct_unknown_quando_sim
0,Education_Level,Marital_Status e Unknown?,14.98,15.22
1,Education_Level,Income_Category e Unknown?,15.24,13.04
2,Marital_Status,Education_Level e Unknown?,7.38,7.50
3,Marital_Status,Income_Category e Unknown?,7.40,7.37
4,Income_Category,Education_Level e Unknown?,11.23,9.55
5,Income_Category,Marital_Status e Unknown?,10.98,10.95


## 3. A taxa de Unknown muda conforme outras variáveis categóricas?

In [6]:
# [IA] Claude Code – % de Unknown em cada coluna, por categoria de outras variaveis
def taxa_unknown_por(variavel):
    tabela = pd.concat([df[variavel], indicadores], axis=1).groupby(variavel).mean() * 100
    tabela.columns = [c.replace("unknown_", "pct_unknown_") for c in tabela.columns]
    tabela["qtd_linhas"] = df[variavel].value_counts()
    return tabela.round(2)

taxa_unknown_por("Attrition_Flag")

,pct_unknown_Education_Level,pct_unknown_Marital_Status,pct_unknown_Income_Category,qtd_linhas
Attrition_Flag,,,,
Attrited Customer,15.73,7.93,11.49,1627
Existing Customer,14.86,7.29,10.88,8500


In [7]:
# [IA] Claude Code – mesma tabela por genero
taxa_unknown_por("Gender")

,pct_unknown_Education_Level,pct_unknown_Marital_Status,pct_unknown_Income_Category,qtd_linhas
Gender,,,,
F,15.15,7.09,19.78,5358
M,14.82,7.74,1.09,4769


In [8]:
# [IA] Claude Code – mesma tabela por categoria do cartao
taxa_unknown_por("Card_Category")

,pct_unknown_Education_Level,pct_unknown_Marital_Status,pct_unknown_Income_Category,qtd_linhas
Card_Category,,,,
Blue,15.05,7.24,11.07,9436
Gold,15.52,10.34,7.76,116
Platinum,15.00,10.00,25.00,20
Silver,14.05,9.37,9.55,555


## 4. Quem tem Unknown é diferente nas variáveis numéricas?

In [9]:
# [IA] Claude Code – mediana das variaveis numericas: linhas com Unknown x linhas com valor conhecido
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM")

comparacoes = {}
for c in COLUNAS_COM_UNKNOWN:
    eh_unknown = df[c] == "Unknown"
    comparacoes[(c, "conhecido")] = df.loc[~eh_unknown, colunas_numericas].median()
    comparacoes[(c, "unknown")] = df.loc[eh_unknown, colunas_numericas].median()

medianas = pd.DataFrame(comparacoes).round(2)
medianas

Education_Level          Marital_Status           \
                               conhecido  unknown      conhecido  unknown   
Customer_Age                       46.00    47.00          46.00    45.00   
Dependent_count                     2.00     2.00           2.00     3.00   
Months_on_book                     36.00    36.00          36.00    36.00   
Total_Relationship_Count            4.00     4.00           4.00     4.00   
Months_Inactive_12_mon              2.00     2.00           2.00     2.00   
Contacts_Count_12_mon               2.00     2.00           2.00     2.00   
Credit_Limit                     4561.50  4411.00        4489.50  5222.00   
Total_Revolving_Bal              1278.00  1270.00        1278.00  1261.00   
Avg_Open_To_Buy                  3480.00  3393.00        3434.50  3983.00   
Total_Amt_Chng_Q4_Q1                0.74     0.74           0.74     0.73   
Total_Trans_Amt                  3906.00  3872.00        3866.00  4182.00   
Total_Trans_Ct                     67.00    67.00          67.00    69.00   
Total_Ct_Chng_Q4_Q1                 0.70     0.71           0.70     0.71   
Avg_Utilization_Ratio               0.18     0.17           0.18     0.15   

                         Income_Category           
                               conhecido  unknown  
Customer_Age                       46.00    46.00  
Dependent_count                     2.00     2.00  
Months_on_book                     36.00    36.00  
Total_Relationship_Count            4.00     4.00  
Months_Inactive_12_mon              2.00     2.00  
Contacts_Count_12_mon               2.00     2.00  
Credit_Limit                     4331.00  6380.00  
Total_Revolving_Bal              1281.00  1243.00  
Avg_Open_To_Buy                  3276.00  5464.50  
Total_Amt_Chng_Q4_Q1                0.74     0.74  
Total_Trans_Amt                  3868.00  4107.00  
Total_Trans_Ct                     67.00    69.00  
Total_Ct_Chng_Q4_Q1                 0.70     0.70  
Avg_Utilization_Ratio               0.18     0.12

In [10]:
# [IA] Claude Code – diferenca relativa (%) da mediana: (unknown - conhecido) / conhecido
diferenca_pct = pd.DataFrame({
    c: ((medianas[(c, "unknown")] - medianas[(c, "conhecido")]) / medianas[(c, "conhecido")] * 100)
    for c in COLUNAS_COM_UNKNOWN
}).round(1)
diferenca_pct

,Education_Level,Marital_Status,Income_Category
Customer_Age,2.2,-2.2,0.0
Dependent_count,0.0,50.0,0.0
Months_on_book,0.0,0.0,0.0
Total_Relationship_Count,0.0,0.0,0.0
Months_Inactive_12_mon,0.0,0.0,0.0
Contacts_Count_12_mon,0.0,0.0,0.0
Credit_Limit,-3.3,16.3,47.3
Total_Revolving_Bal,-0.6,-1.3,-3.0
Avg_Open_To_Buy,-2.5,16.0,66.8
Total_Amt_Chng_Q4_Q1,0.0,-1.4,0.0


## 5. Mecanismo de ausência

Para Education_Level, o mecanismo mais próximo parece ser MCAR, pois as taxas de Unknown apresentam poucas diferenças entre os grupos analisados. Para Marital_Status, existem algumas diferenças conforme outras variáveis, então podemos considerar a possibilidade de MAR. Já em Income_Category, a ausência apresenta uma diferença muito grande entre homens e mulheres, indicando um comportamento mais próximo de MAR. Não é possível afirmar esses mecanismos com certeza apenas pelas tabelas, principalmente porque não conseguimos observar diretamente o motivo pelo qual cada cliente deixou de informar seus dados.


## 6. Tratamento

In [11]:
# [IA] Claude Code – aplicar o tratamento escolhido pelo grupo
# Opcoes:
#   "categoria_propria": mantem "Unknown" como uma categoria a parte (nenhuma linha e removida
#                        e nenhum valor e inventado)
#   "moda":              substitui "Unknown" pela categoria mais frequente da coluna
# O enunciado nao permite simplesmente remover as linhas com ausencia.
TRATAMENTO = "categoria_propria"  # (decisao do grupo)

df_tratado = df.copy()
if TRATAMENTO == "categoria_propria":
    pass
elif TRATAMENTO == "moda":
    for c in COLUNAS_COM_UNKNOWN:
        moda = df.loc[df[c] != "Unknown", c].mode()[0]
        df_tratado[c] = df_tratado[c].replace("Unknown", moda)
else:
    raise ValueError(f"Tratamento desconhecido: {TRATAMENTO}")

# Conferencia: nenhuma linha foi removida
assert len(df_tratado) == len(df)
pd.DataFrame({c: df_tratado[c].value_counts() for c in COLUNAS_COM_UNKNOWN}).fillna(0).astype(int)

,Education_Level,Marital_Status,Income_Category
$120K +,0,0,727
$40K - $60K,0,0,1790
$60K - $80K,0,0,1402
$80K - $120K,0,0,1535
College,1013,0,0
Divorced,0,748,0
Doctorate,451,0,0
Graduate,3128,0,0
High School,2013,0,0
Less than $40K,0,0,3561


In [12]:
# [IA] Claude Code – salvar o dataset tratado para as proximas etapas (EDA e dashboard)
PASTA_SAIDA = Path("../../out/parte_especifica")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
df_tratado.to_csv(PASTA_SAIDA / "bankchurners_tratado.csv", index=False)
print("Salvo em", PASTA_SAIDA / "bankchurners_tratado.csv")

Salvo em ..\..\out\parte_especifica\bankchurners_tratado.csv


### Justificativa do tratamento

Decidimos manter Unknown como uma categoria própria porque não sabemos qual seria o valor correto de cada informação ausente. Substituir pela moda poderia criar informações que não correspondem à realidade dos clientes. Além disso, manter essa categoria permite analisar os registros sem excluir clientes ou perder informações importantes.
